In [ ]:
#All the imports
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import norm
from scipy.stats import kurtosis
from scipy.ndimage import maximum_filter, label
import multiprocessing as mp
import os
import time

# shared code and paths (see README.md and shared/)
import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "paths.py").exists())
sys.path.insert(0, str(REPO))
from shared.paths import DATA_ROOT, NAS_ROOT, EXTERNAL_DATA_ROOT, results_dir, cache_dir as shared_cache_dir
RESULTS = results_dir("quality_checks", "Jrms")

In [ ]:
#Function Definitions
def partial_x_2d(f,dx):
    """Compute ∂f/∂x using 2nd-order central differences with periodic BCs."""
    Nx, Ny = f.shape
    df_dx = np.zeros_like(f)
    
    # Central difference in x-direction (axis=0)
    df_dx[1:Nx-1, :] = (f[2:Nx, :] - f[0:Nx-2, :]) / (2 * dx)
    
    # Periodic boundaries (x-direction)
    df_dx[0, :] = (f[1, :] - f[-1, :]) / (2 * dx)
    df_dx[-1, :] = (f[0, :] - f[-2, :]) / (2 * dx)
    
    return df_dx

def partial_y_2d(f,dy):
    """Compute ∂f/∂y using 2nd-order central differences with periodic BCs."""
    Nx, Ny = f.shape
    df_dy = np.zeros_like(f)
    
    # Central difference in y-direction (axis=1)
    df_dy[:, 1:Ny-1] = (f[:, 2:Ny] - f[:, 0:Ny-2]) / (2 * dy)
    
    # Periodic boundaries (y-direction)
    df_dy[:, 0] = (f[:, 1] - f[:, -1]) / (2 * dy)
    df_dy[:, -1] = (f[:, 0] - f[:, -2]) / (2 * dy)
    
    return df_dy

def partial_x_fft(f, dx):
    """
    Calculate ∂f/∂x using FFT on a 2D grid.

    Parameters:
        f (2D np.array): Function values on a 2D grid (shape: Nx × Ny).
        dx (float): Grid spacing in the x-direction.

    Returns:
        df_dx (2D np.array): Partial derivative ∂f/∂x.
    """
    nx, ny = f.shape
    kx = np.fft.fftfreq(nx, d=dx) * 2 * np.pi  # Wavenumbers in x
    kx = kx[:, np.newaxis]                     # Make it 2D for broadcasting

    f_hat = np.fft.fft2(f)
    df_dx_hat = 1j * kx * f_hat
    df_dx = np.fft.ifft2(df_dx_hat).real

    return np.array(df_dx)

def partial_y_fft(f, dy):
    """
    Calculate ∂f/∂y using FFT on a 2D grid.

    Parameters:
        f (2D np.array): Function values on a 2D grid (shape: Nx × Ny).
        dy (float): Grid spacing in the y-direction.

    Returns:
        df_dy (2D np.array): Partial derivative ∂f/∂y.
    """
    nx, ny = f.shape
    ky = np.fft.fftfreq(ny, d=dy) * 2 * np.pi  # Wavenumbers in y
    ky = ky[np.newaxis, :]                     # Make it 2D for broadcasting

    f_hat = np.fft.fft2(f)
    df_dy_hat = 1j * ky * f_hat
    df_dy = np.fft.ifft2(df_dy_hat).real

    return df_dy

def index_of_just_smaller(arr, value):
    arr = np.asarray(arr)
    mask = arr < value
    if not np.any(mask):
        return None  # No smaller value exists
    return np.argmax(np.where(mask, arr, -np.inf))

#Denoising the data
def gaussian_fft_filter_2d(J_squared, k0):
    """
    Apply a Gaussian low-pass filter in Fourier space to a 2D field.

    Parameters
    ----------
    J_squared : 2D numpy array
        Input real-space data
    k0 : float
        Filter width in k-space (grid units). Smaller = stronger smoothing.

    Returns
    -------
    J_filtered : 2D numpy array
        Smoothed real-space data
    """

    # FFT
    F = np.fft.fft2(J_squared)
    F_shifted = np.fft.fftshift(F)

    # k-grid
    nx, ny = J_squared.shape
    kx = np.fft.fftshift(np.fft.fftfreq(nx)) * nx
    ky = np.fft.fftshift(np.fft.fftfreq(ny)) * ny
    KX, KY = np.meshgrid(kx, ky, indexing='ij')
    K2 = KX**2 + KY**2

    # Gaussian filter in k-space
    gaussian = np.exp(-K2 / (2 * k0**2))

    # Apply filter
    F_filt = F_shifted * gaussian

    # Inverse FFT
    F_filt = np.fft.ifftshift(F_filt)
    J_filtered = np.fft.ifft2(F_filt)

    # Return real part
    return np.real(J_filtered)

In [ ]:
# List of simulation directories
sim_dirs = [
    f'{DATA_ROOT}/beta_p_1/beta_e_1/'
]

dx = 0.0625
dy = 0.0625

In [ ]:
def process_timestep(args):
    sim_dir, i = args
    
    # print(f"{sim_dir}: i = {i}", flush=True)

    File_Bx = f'{sim_dir}/Bx_ApJ_t{i}.h5'
    File_By = f'{sim_dir}/By_ApJ_t{i}.h5'

    with h5py.File(File_Bx, 'r') as fBx, h5py.File(File_By, 'r') as fBy:
        data_Bx = fBx['DS1'][:].T
        data_By = fBy['DS1'][:].T

    # Filtering 
    
    data_Bx_filtered = gaussian_fft_filter_2d(data_Bx, k0=0.2 * len(data_Bx))
    data_By_filtered = gaussian_fft_filter_2d(data_By, k0=0.2 * len(data_Bx))

    data_Bx = data_Bx_filtered
    data_By = data_By_filtered
    
    J_par_FT = partial_x_fft(data_By, dx) - partial_y_fft(data_Bx, dy)
    rms = np.sqrt(np.mean(J_par_FT**2))

    return i, rms


def process_sim_dir(sim_dir):
    timesteps = [(sim_dir, i) for i in range(300)]

    start = time.perf_counter()
    with mp.Pool(processes=128) as pool:
        results = pool.map(process_timestep, timesteps)
    end = time.perf_counter()

    print(f"Total runtime: {end - start:.2f} seconds")
    # sort by timestep
    results.sort()
    J_rms = [rms for _, rms in results]

    return sim_dir, J_rms


if __name__ == "__main__":
    all_J_rms_filter = {}

    for sim_dir in sim_dirs:
        sim_dir, J_rms = process_sim_dir(sim_dir)
        all_J_rms_filter[sim_dir] = J_rms

In [ ]:
timesteps = np.arange(0, 300)

plt.figure(figsize=(10, 6))

# No filter
# for sim_dir, J_rms in all_J_rms_nofilter.items():
#     label = sim_dir.split('/')[-1] + " (no filter)"
#     plt.plot(timesteps, J_rms, label=label)

# Filtered
for sim_dir, J_rms in all_J_rms_filter.items():
    label = sim_dir.split('/')[-1] + " (filtered)"
    plt.plot(timesteps, J_rms, '--', label=label)  # dashed for filtered

plt.xlabel('Timestep')
plt.ylabel(r'$J_{rms}$')
plt.title('J_rms vs Time for Different Simulations')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(RESULTS / "Jrms.png", dpi=300)
plt.show()

In [ ]:
np.max(J_rms)

In [ ]:
import numpy as np

J_rms = np.array(J_rms)
timesteps = np.array(timesteps)

J_max = np.max(J_rms)
threshold = 0.95 * J_max

indices = np.where(J_rms >= threshold)[0]
times_above = timesteps[indices]
values_above = J_rms[indices]

In [ ]:
print(indices)